# InsightForge: AI-Powered Multi-Agent Analytics Platform

This notebook demonstrates the multi-agent analytics pipeline built with **AutoGen** and **Claude AI (Anthropic)**.

The system uses specialized AI agents that collaborate to analyze datasets:
- **Admin Agent** — Coordinates the analysis pipeline
- **Data Quality Agent** — Checks for missing values, duplicates, and data types
- **EDA Agent** — Computes statistics, correlations, and distributions
- **Insight Agent** — Derives actionable business insights
- **Visualization Agent** — Generates charts and plots
- **Report Agent** — Compiles the final deliverable

## 1. Setup & Dependencies

In [ ]:
!pip install autogen-agentchat autogen-ext[anthropic] pandas matplotlib -q

In [ ]:
import os
import asyncio
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat, SelectorGroupChat
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_ext.models.anthropic import AnthropicChatCompletionClient
from autogen_core.models import AssistantMessage, LLMMessage
from typing import Sequence

## 2. Configure the LLM Client

We use Claude Sonnet 4.6 via the Anthropic API. A patched client is needed to handle AutoGen's multi-turn conversation format.

In [ ]:
# Set your API key
os.environ["ANTHROPIC_API_KEY"] = "your-api-key-here"  # Replace with your key


class PatchedAnthropicClient(AnthropicChatCompletionClient):
    """Patched client to fix prefill issue with Claude Sonnet 4.6.
    
    AutoGen sends full conversation history ending with an assistant message,
    which Claude Sonnet 4.6 rejects. This patch strips trailing assistant messages.
    """
    def _rstrip_last_assistant_message(self, messages: Sequence[LLMMessage]) -> Sequence[LLMMessage]:
        msgs = list(messages)
        while msgs and isinstance(msgs[-1], AssistantMessage):
            msgs.pop()
        return msgs


model_client = PatchedAnthropicClient(
    model="claude-sonnet-4-6",
    api_key=os.environ["ANTHROPIC_API_KEY"],
    max_tokens=4096,
)

print("Model client configured successfully.")

## 3. Load Sample Dataset

In [ ]:
# Create a sample dataset
data = {
    "id": range(1, 21),
    "name": ["Alice Johnson", "Bob Smith", "Charlie Brown", "Diana Ross", "Eve Davis",
             "Frank Miller", "Grace Lee", "Henry Wilson", "Iris Chen", "Jack Taylor",
             "Karen White", "Leo Martinez", "Mona Patel", "Nathan Kim", "Olivia Brown",
             "Paul Anderson", "Quinn Roberts", "Rachel Green", "Sam Thompson", "Tina Nguyen"],
    "age": [34, 28, 45, 31, 29, 52, 38, 41, 26, 35, 44, 33, 29, 47, 36, 30, 42, 27, 39, 34],
    "email": ["alice@example.com", "bob@example.com", "charlie@example.com", None,
              "eve@example.com", "frank@example.com", "grace@example.com", "henry@example.com",
              "iris@example.com", "jack@example.com", "karen@example.com", "leo@example.com",
              "mona@example.com", "nathan@example.com", None, "paul@example.com",
              "quinn@example.com", "rachel@example.com", "sam@example.com", "tina@example.com"],
    "salary": [75000, 55000, 90000, 60000, 65000, 105000, None, 87000, 54000, 72000,
               91000, 67000, 59000, 98000, 71000, 63000, 88000, 55000, 82000, 74000],
    "department": ["Engineering", "Marketing", "Engineering", "Sales", "Marketing",
                   "Engineering", "HR", "Sales", "Marketing", "Engineering",
                   "HR", "Sales", "Marketing", "Engineering", "Sales",
                   "HR", "Engineering", "Marketing", "Sales", "Engineering"],
    "hire_date": ["2019-03-15", "2020-07-01", "2015-11-20", "2021-01-10", "2020-09-14",
                  "2012-06-03", "2018-04-22", "2017-08-30", "2022-02-14", "2019-10-01",
                  "2016-05-18", "2020-03-25", "2021-06-07", "2014-09-12", "2018-12-01",
                  "2021-08-19", "2016-02-28", "2022-05-10", "2017-11-15", "2019-07-22"],
    "is_active": [True, True, True, True, False, True, True, True, True, True,
                  True, False, True, True, True, True, True, True, True, True],
}

df = pd.DataFrame(data)
DATA_PATH = "sample_dataset.csv"
df.to_csv(DATA_PATH, index=False)

print(f"Dataset: {len(df)} rows x {len(df.columns)} columns")
df.head()

## 4. Define Agent Tools

Each tool accepts a file path (JSON-serializable) so the LLM can call it via AutoGen's function calling.

In [ ]:
def analyze_data_quality(file_path: str) -> str:
    """Analyze dataset quality. Returns a summary of missing values, duplicates, and column types."""
    df = pd.read_csv(file_path)
    total_records = len(df)
    total_columns = len(df.columns)
    missing_values = int(df.isnull().sum().sum())
    duplicate_rows = int(df.duplicated().sum())
    column_types = {col: str(dtype) for col, dtype in df.dtypes.items()}

    return (
        f"Data Quality Report:\n"
        f"- Total Records: {total_records}\n"
        f"- Total Columns: {total_columns}\n"
        f"- Missing Values: {missing_values}\n"
        f"- Duplicate Rows: {duplicate_rows}\n"
        f"- Column Types: {column_types}"
    )


def run_eda(file_path: str) -> str:
    """Run exploratory data analysis. Returns summary statistics and correlations."""
    df = pd.read_csv(file_path)
    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    categorical_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()

    stats = {col: df[col].describe().to_dict() for col in numeric_cols}
    correlations = df[numeric_cols].corr().to_dict() if numeric_cols else {}

    result = f"EDA Results:\n"
    result += f"- Numeric Columns: {numeric_cols}\n"
    result += f"- Categorical Columns: {categorical_cols}\n\n"
    result += "Summary Statistics:\n"
    for col, s in stats.items():
        result += f"  {col}: mean={s.get('mean', 'N/A'):.2f}, std={s.get('std', 'N/A'):.2f}, min={s.get('min', 'N/A')}, max={s.get('max', 'N/A')}\n"
    result += f"\nCorrelation (age vs salary): {correlations.get('salary', {}).get('age', 'N/A'):.3f}"
    return result


def generate_visualizations(file_path: str) -> str:
    """Generate histogram visualizations for numeric columns. Returns list of generated chart paths."""
    df = pd.read_csv(file_path)
    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    charts = []

    for col in numeric_cols:
        if col == "id":
            continue
        plt.figure(figsize=(8, 4))
        plt.hist(df[col].dropna(), bins=15, alpha=0.7, color="#6366f1", edgecolor="white")
        plt.title(f"Distribution of {col}", fontsize=12, fontweight="bold")
        plt.xlabel(col)
        plt.ylabel("Frequency")
        plt.grid(axis="y", alpha=0.3)
        path = f"{col}_histogram.png"
        plt.savefig(path, dpi=100, bbox_inches="tight")
        plt.close()
        charts.append(path)

    return f"Generated {len(charts)} charts: {', '.join(charts)}"


print("Tools defined successfully.")

## 5. Define AI Agents

Each agent has a specific role and access to relevant tools.

In [ ]:
admin_agent = AssistantAgent(
    name="admin_agent",
    model_client=model_client,
    description="The Analytics Team Lead responsible for coordinating agents and delivering insights.",
    system_message="""
    You are the Analytics Team Lead at InsightForge.
    Coordinate the data analysis pipeline:
    1. Direct the Data Quality Agent to assess dataset health
    2. Direct the EDA Agent to explore patterns
    3. Direct the Insight Agent to derive business insights
    4. Direct the Visualization Agent to generate charts
    Summarize the overall findings at the end.
    """,
)

data_quality_agent = AssistantAgent(
    name="data_quality_agent",
    model_client=model_client,
    tools=[analyze_data_quality],
    description="Analyzes dataset quality by checking for missing values, duplicates, and data types.",
    system_message="""
    You are the Data Quality Agent. Use the analyze_data_quality tool with the file path
    to check for missing values, duplicates, and data types. Summarize findings clearly.
    """,
)

eda_agent = AssistantAgent(
    name="eda_agent",
    model_client=model_client,
    tools=[run_eda],
    description="Performs exploratory data analysis including summary statistics and correlations.",
    system_message="""
    You are the EDA Agent. Use the run_eda tool to compute summary statistics,
    identify patterns, and calculate correlations. Highlight interesting findings.
    """,
)

insight_agent = AssistantAgent(
    name="insight_agent",
    model_client=model_client,
    description="Derives actionable business insights from the analysis results.",
    system_message="""
    You are the Insight Agent. Analyze outputs from other agents and derive
    actionable business insights. Focus on trends, anomalies, and recommendations.
    """,
)

visualization_agent = AssistantAgent(
    name="visualization_agent",
    model_client=model_client,
    tools=[generate_visualizations],
    description="Generates histogram visualizations for numeric columns.",
    system_message="""
    You are the Visualization Agent. Use the generate_visualizations tool
    to create charts for the dataset. Report which charts were generated.
    """,
)

print("All agents created successfully.")
print(f"  - admin_agent (coordinator)")
print(f"  - data_quality_agent (tool: analyze_data_quality)")
print(f"  - eda_agent (tool: run_eda)")
print(f"  - insight_agent (reasoning only)")
print(f"  - visualization_agent (tool: generate_visualizations)")

## 6. Create the Multi-Agent Team

We use a `RoundRobinGroupChat` where agents take turns analyzing the dataset.

In [ ]:
termination = MaxMessageTermination(max_messages=15)

analytics_team = RoundRobinGroupChat(
    participants=[
        data_quality_agent,
        eda_agent,
        visualization_agent,
        insight_agent,
    ],
    termination_condition=termination,
)

print("Analytics team assembled.")
print(f"Team type: {type(analytics_team).__name__}")
print(f"Max messages: 15")

## 7. Run the Analytics Pipeline

The agents collaborate to analyze the dataset end-to-end.

In [ ]:
async def run_pipeline():
    task = f"Analyze the dataset at {DATA_PATH}. Check data quality, run EDA, generate visualizations, and provide business insights."
    result = await analytics_team.run(task=task)
    return result

result = await run_pipeline()

print(f"\n{'='*60}")
print(f"Pipeline complete! Total messages: {len(result.messages)}")
print(f"{'='*60}")

## 8. View Agent Conversation

In [ ]:
for msg in result.messages:
    source = msg.source
    content = str(msg.content)
    print(f"\n{'─'*60}")
    print(f"🤖 [{source}]")
    print(f"{'─'*60}")
    print(content[:500])
    if len(content) > 500:
        print("...")

## 9. Display Generated Visualizations

In [ ]:
chart_files = list(Path(".").glob("*_histogram.png"))

if chart_files:
    fig, axes = plt.subplots(1, len(chart_files), figsize=(6 * len(chart_files), 4))
    if len(chart_files) == 1:
        axes = [axes]
    for ax, chart in zip(axes, sorted(chart_files)):
        img = plt.imread(str(chart))
        ax.imshow(img)
        ax.set_title(chart.stem.replace("_", " ").title(), fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No charts generated yet — run the pipeline first.")

## 10. Direct Tool Execution (without agents)

The tools can also be called directly for quick analysis without the multi-agent orchestration.

In [ ]:
print("=" * 50)
print("DIRECT TOOL EXECUTION")
print("=" * 50)

print("\n📊 Data Quality:")
print(analyze_data_quality(DATA_PATH))

print("\n📈 EDA:")
print(run_eda(DATA_PATH))

print("\n🎨 Visualizations:")
print(generate_visualizations(DATA_PATH))

## 11. Architecture Overview

```
┌─────────────────────────────────────────────────────────────┐
│                    InsightForge Platform                      │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│  Frontend (React + Vite)                                     │
│  ├── Chat Interface (upload CSV + ask questions)             │
│  ├── Dashboard (KPIs + recent analyses)                      │
│  └── Analysis Pipeline View                                  │
│                                                              │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│  Backend (FastAPI)                                           │
│  ├── /api/chat — Chat with AI agents about your data        │
│  ├── /api/datasets/upload — Upload CSV files                 │
│  └── /api/analysis/run — Direct tool execution               │
│                                                              │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│  AI Agent Layer (AutoGen + Claude)                           │
│  ├── Admin Agent ──────── Coordinates pipeline               │
│  ├── Data Quality Agent ─ analyze_data_quality()             │
│  ├── EDA Agent ────────── run_eda()                          │
│  ├── Insight Agent ────── Pure LLM reasoning                 │
│  ├── Visualization Agent─ generate_visualizations()          │
│  └── Report Agent ────── generate_report()                   │
│                                                              │
└─────────────────────────────────────────────────────────────┘
```

## 12. Key Technical Decisions

| Decision | Choice | Rationale |
|----------|--------|-----------|
| LLM Provider | Claude (Anthropic) | Strong reasoning, tool use support |
| Agent Framework | AutoGen | Multi-agent orchestration with tool calling |
| Team Type | RoundRobinGroupChat | Predictable agent execution order |
| Tool Design | File-path based | JSON-serializable inputs for LLM compatibility |
| Frontend | React + Vite | Fast dev, modern UI |
| Backend | FastAPI | Async-native, auto-generated docs |
| Chat | Direct Claude API | Avoids multi-agent overhead for simple Q&A |

---

**End of Notebook**

This demonstrates the core multi-agent analytics pipeline. The full application adds a React frontend with a chat interface, file upload, and markdown-rendered responses.